In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [2]:
from crewai import Agent, Task, Crew

In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
load_dotenv(env_path, override=True)

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise RuntimeError(f"GROQ_API_KEY not found in {env_path} (cwd={Path.cwd()})")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="qwen/qwen3.8-27b",
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_api_key,
    temperature=0.7,
    max_tokens=800,
    max_retries=5,
    request_timeout=120,
)

In [4]:
from utils import get_serper_api_key

serper_api_key = get_serper_api_key()
if not serper_api_key:
    raise RuntimeError(f"SERPER_API_KEY not found in {env_path} (cwd={Path.cwd()})")

os.environ["SERPER_API_KEY"] = serper_api_key

In [5]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

In [6]:
data_analyst_agent = Agent(
    role="Data Analyst",
    goal="Monitor and analyze market data in real-time "
    "to identify trends and predict market movements.",
    backstory="Specializing in financial markets, this agent "
    "uses statistical modeling and machine learning "
    "to provide crucial insights. With a knack for data, "
    "the Data Analyst Agent is the cornerstone for "
    "informing trading decisions.",
    allow_delegation=True,
    verbose=True,
    tools=[search_tool, scrape_tool],
    llm=llm,
)

In [7]:
trading_strategy_agent = Agent(
    role="Trading Strategy Developer",
    goal="Develop and test various trading strategies based "
    "on insights from the Data Analyst Agent.",
    backstory="Equipped with a deep understanding of financial "
    "markets and quantitative analysis, this agent "
    "devises and refines trading strategies. It evaluates "
    "the performance of different approaches to determine "
    "the most profitable and risk-averse options.",
    verbose=True,
    allow_delegation=True,
    tools=[scrape_tool, search_tool],
    llm=llm,
)

In [8]:
execution_agent = Agent(
    role="Trade Advisor",
    goal="Suggest optimal trade execution strategies "
    "based on approved trading strategies.",
    backstory="This agent specializes in analyzing the timing, price, "
    "and logistical details of potential trades. By evaluating "
    "these factors, it provides well-founded suggestions for "
    "when and how trades should be executed to maximize "
    "efficiency and adherence to strategy.",
    verbose=True,
    allow_delegation=True,
    tools=[scrape_tool, search_tool],
    llm=llm,
)

In [9]:
risk_management_agent = Agent(
    role="Risk Advisor",
    goal="Evaluate and provide insights on the risks "
    "associated with potential trading activities.",
    backstory="Armed with a deep understanding of risk assessment models "
    "and market dynamics, this agent scrutinizes the potential "
    "risks of proposed trades. It offers a detailed analysis of "
    "risk exposure and suggests safeguards to ensure that "
    "trading activities align with the firm’s risk tolerance.",
    verbose=True,
    allow_delegation=True,
    tools=[scrape_tool, search_tool],
    llm=llm,
)

In [10]:
data_analysis_task = Task(
    description=(
        "Continuously monitor and analyze market data for "
        "the selected stock ({stock_selection}). "
        "Use statistical modeling and machine learning to "
        "identify trends and predict market movements."
    ),
    expected_output=(
        "Insights and alerts about significant market "
        "opportunities or threats for {stock_selection}."
    ),
    agent=data_analyst_agent,
)

In [11]:
strategy_development_task = Task(
    description=(
        "Develop and refine trading strategies based on "
        "the insights from the Data Analyst and "
        "user-defined risk tolerance ({risk_tolerance}). "
        "Consider trading preferences ({trading_strategy_preference})."
    ),
    expected_output=(
        "A set of potential trading strategies for {stock_selection} "
        "that align with the user's risk tolerance."
    ),
    agent=trading_strategy_agent
)

In [12]:
execution_planning_task = Task(
    description=(
        "Analyze approved trading strategies to determine the "
        "best execution methods for {stock_selection}, "
        "considering current market conditions and optimal pricing."
    ),
    expected_output=(
        "Detailed execution plans suggesting how and when to "
        "execute trades for {stock_selection}."
    ),
    agent=execution_agent
)

In [13]:
risk_assessment_task = Task(
    description=(
        "Evaluate the risks associated with the proposed trading "
        "strategies and execution plans for {stock_selection}. "
        "Provide a detailed analysis of potential risks "
        "and suggest mitigation strategies."
    ),
    expected_output=(
        "A comprehensive risk analysis report detailing potential "
        "risks and mitigation recommendations for {stock_selection}."
    ),
    agent=risk_management_agent
)

In [16]:
from crewai import Process

financial_trading_crew = Crew(
    agents=[
        data_analyst_agent,
        trading_strategy_agent,
        execution_agent,
        risk_management_agent,
    ],
    tasks=[
        data_analysis_task,
        strategy_development_task,
        execution_planning_task,
        risk_assessment_task,
    ],
    manager_llm=llm,
    process=Process.hierarchical,
    verbose=True,
)

2026-09-21 14:51:26,840 - 26016 - __init__.py-__init__:556 - WARNING: Overriding of current TracerProvider is not allowed


In [ ]:
financial_trading_inputs = {
    "stock_selection": "AAPL",
    "initial_capital": "100000",
    "risk_tolerance": "medium",
    "trading_strategy_preference": "Day Trading",
    "news_impact_consideration": True,
}

result = financial_trading_crew.kickoff(inputs=financial_trading_inputs)

 [DEBUG]: Working Agent: Crew Manager
 [INFO]: Starting Task: Continuously monitor and analyze market data for the selected stock (AAPL). Use statistical modeling and machine learning to identify trends and predict market movements.


> Entering new CrewAgentExecutor chain...
Thought: To achieve the goal of continuously monitoring and analyzing market data for AAPL using statistical modeling and machine learning, I need to break this down into two primary components: data acquisition/processing and predictive modeling. First, I need to ensure we have robust, clean data to work with. The Data Analyst is the right person to handle the collection, cleaning, and initial statistical analysis of AAPL's market data. I will delegate this task to them, specifying the need for historical price data, volume, and relevant technical indicators, as well as any data quality checks.

Action: Delegate work to co-worker
Action Input: {"coworker": "Data Analyst", "task": "Collect, clean, and prepare comp

2026-09-21 14:54:34,428 - 25060 - __init__.py-__init__:216 - ERROR: Failed to export span batch code: None, reason: HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Read timed out. (read timeout=30.0)


Thought: I have gathered technical indicators from Financhill. I need to gather more specific OHLCV data and potentially fundamental data to complete the dataset. I will search for more detailed historical data or a source that provides a table of recent OHLCV values.
Action: Search the internet
Action Input: {"search_query": "AAPL historical stock price data OHLCV last 10 days table"} 


Search results: Title: Apple Inc. (AAPL)
Link: https://finance.yahoo.com/quote/AAPL/history/
Snippet: Apple Inc. (AAPL) ; Sep 14, 2026, 334.79, 335.50 ; Sep 11, 2026, 327.45, 336.22 ; Sep 10, 2026, 316.67, 326.74 ; Sep 9, 2026, 315.49, 319.15 ...
---
Title: Apple Inc. Common Stock (AAPL) Historical Quotes
Link: https://www.nasdaq.com/market-activity/stocks/aapl/historical
Snippet: Find the latest historical data for Apple Inc. Common Stock (AAPL) at Nasdaq.com. View historical data in a monthly, bi-annual, or yearly format.
---
Title: Apple - 46 Year Stock Price History | AAPL
Link: https://www.macrot

2026-09-21 14:55:09,641 - 25060 - __init__.py-__init__:216 - ERROR: Failed to export span batch code: None, reason: HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Read timed out. (read timeout=30.0)
2026-09-21 14:55:44,848 - 25060 - __init__.py-__init__:216 - ERROR: Failed to export span batch code: None, reason: HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Read timed out. (read timeout=30.0)


Thought: I need to gather comprehensive market data for Apple Inc. (AAPL). Since I cannot directly access a live financial data API, I will search the internet for recent historical data, technical indicators, and fundamental information for AAPL. I will start by searching for recent daily OHLCV data and technical analysis summaries.

Action: Search the internet
Action Input: {"search_query": "AAPL Apple Inc. recent daily OHLCV data technical indicators RSI MACD Bollinger Bands 2023 2024"} 


Search results: Title: Apple, Inc. (AAPL) Technical Analysis - Stock Price Charts
Link: https://financhill.com/stock-price-chart/aapl-technical-analysis
Snippet: Apple, Inc. 50-day exponential moving average is 317.69 while AAPL share price is 336.13, making it a Buy technically. AAPL stock price is 336.13 and ...
---
Title: AAPL Technical Analysis, RSI and Moving Averages
Link: https://www.investing.com/equities/apple-computer-inc-technical
Snippet: The current Relative Strength Index (RSI) of AA

...

Thought: The Macroaxis page returned a garbled or empty response. I need to try another source. The Barchart link is a standard financial data provider and often has accessible technical analysis summaries. I will also perform a more specific search for "AAPL current volatility and volume anomaly" to see if a news source has highlighted any immediate data anomalies.

Action: Search the internet
Action Input: {"search_query": "AAPL stock today volume anomaly volatility RSI MACD latest analysis"} 


Search results: Title: Apple Inc. (AAPL)
Link: https://finance.yahoo.com/quote/AAPL/
Snippet: Apple's latest news centers on a major foldable iPhone launch, a reported push into AI servers, and a Fitness+ reset that may trim costs and ...
---
Title: AAPL Technical Analysis | Trend, Signals & Chart Patterns
Link: https://www.chartmill.com/stock/quote/AAPL/technical-analysis
Snippet: volatility and chart patterns. AAPL Stock Price Today 335.26 USD -1.74 (-0.52%) volatility and current chart pat

2026-09-21 15:21:25,005 - 25060 - __init__.py-__init__:216 - ERROR: Failed to export span batch code: None, reason: HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Read timed out. (read timeout=30.0)
2026-09-21 15:22:00,313 - 25060 - __init__.py-__init__:216 - ERROR: Failed to export span batch code: None, reason: HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Read timed out. (read timeout=30.0)


Thought: The Macroaxis page returned an empty or unreadable result. I need to try another source. The Barchart link or the Investtech link might be more accessible and provide the specific statistical indicators like RSI, MACD, and volatility that I need. Let's try the Investtech page, as it explicitly mentions RSI, MACD, and Pivot points in the snippet.

Action: Read website content
Action Input: {"website_url": "https://www.investtech.com/main/market.php?CompanyID=10503307"} 

Apple (AAPL) - Technical Analysis - US Stocks - Investtech
☰ Home
About InvesttechProductsResearchInsight & Skills 
US Stocks ▾Oslo Børs
Stockholmsbörsen
Københavns Fondsbørs
Investeringsforeninger
Helsingin pörssi
World Indices
US Stocks
US 30
Nasdaq 100
US 500
Toronto Stock Exchange
Euronext Amsterdam
Euronext Brussel
DAX
Prime Standard
TECDAX
SDAX
MDAX
CDAX
CAC 40
Mumbai S.E.
National S.E.
Commodities
Currency
Cryptocurrency
Investtech Indices
Exchange Traded Funds
| LoginFree TrialAnalysesStocksFree Trial
A

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3.8-27b` in organization `org_01m2wtajwgem9r6g7bvbprbjjn` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 198955, Requested 2517. Please try again in 10m35.904s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

2026-09-21 15:22:35,578 - 25060 - __init__.py-__init__:216 - ERROR: Failed to export span batch code: None, reason: HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Read timed out. (read timeout=30.0)
2026-09-21 15:23:10,855 - 25060 - __init__.py-__init__:216 - ERROR: Failed to export span batch code: None, reason: HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Read timed out. (read timeout=30.0)
